# 01 Data, Embeddings, SAE

## Setup

In [4]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ['GE_DRIVE_ROOT'] = '/content/drive/MyDrive/hypothesaes_goemotions'
REPO_URL = 'https://github.com/vankadaruvijayberk/HypotheSAEs.git'
REPO_DIR = '/content/HypotheSAEs'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Install

In [5]:
import os, subprocess
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
subprocess.run(['pip', 'install', '-q', '-e', REPO_DIR], check=True)
subprocess.run(['pip', 'install', '-q', '-U', 'datasets', 'sentence-transformers',
                'transformers', 'scikit-learn', 'statsmodels'], check=True)
# editable-install .pth files are only read at interpreter startup, so add the paths directly
import sys, importlib
for p in (REPO_DIR, os.path.join(REPO_DIR, 'goemotions')):
    if p not in sys.path:
        sys.path.insert(0, p)
importlib.invalidate_caches()

## Imports

In [6]:
import numpy as np
import torch
import ge_pipeline as ge
from hypothesaes.embedding import get_local_embeddings
from hypothesaes.quickstart import train_sae
ge.set_seed()
if not torch.cuda.is_available():
    print('WARNING: no GPU')

## Preflight

In [7]:
import inspect
from dataclasses import fields
from hypothesaes import llm_api, embedding, annotate as _annot
from hypothesaes.select_neurons import select_neurons
from hypothesaes.interpret_neurons import (NeuronInterpreter, InterpretConfig,
                                           SamplingConfig, LLMConfig, ScoringConfig)
from hypothesaes.annotate import annotate_texts_with_concepts
from hypothesaes.evaluation import score_hypotheses

assert 'use_chat' in inspect.getsource(llm_api.get_completion), 'fork missing chat-completions patch'
assert 'nomic_prefix' in inspect.signature(embedding.get_local_embeddings).parameters, 'fork missing nomic_prefix patch'
assert {'interpreter_model', 'annotator_model', 'n_workers_interpretation',
        'n_workers_annotation', 'cache_name'} <= set(inspect.signature(NeuronInterpreter.__init__).parameters)
assert {'use_cache_only', 'uncached_value'} <= set(inspect.signature(_annot.annotate).parameters)
assert {'activations', 'target', 'n_select', 'method', 'classification'} <= set(inspect.signature(select_neurons).parameters)
assert 'n_candidates' in {f.name for f in fields(InterpretConfig)}

ge.redirect_annotation_cache()
import hypothesaes.interpret_neurons as _interp
assert _annot.CACHE_DIR == ge.ANNOT_DIR and _interp.CACHE_DIR == ge.ANNOT_DIR
print('preflight OK')

preflight OK


## Load Data

In [8]:
ds, NAMES = ge.load_goemotions()
train_texts = list(ds['train']['text'])
val_texts   = list(ds['validation']['text'])
test_texts  = list(ds['test']['text'])
train_targets = ge.build_targets(ds['train']['labels'], NAMES)
val_targets   = ge.build_targets(ds['validation']['labels'], NAMES)

ge.log_json('01_data', {
    'n_train': len(train_texts), 'n_val': len(val_texts), 'n_test': len(test_texts),
    'positives_train': {t: int(v.sum()) for t, v in train_targets.items()},
    'prevalence_train': {t: float(v.mean()) for t, v in train_targets.items()},
})
{t: int(v.sum()) for t, v in train_targets.items()}

README.md:   0%|          | 0.00/9.40k [00:00<?, ?B/s]

simplified/train-00000-of-00001.parquet:   0%|          | 0.00/2.77M [00:00<?, ?B/s]

simplified/validation-00000-of-00001.par(…):   0%|          | 0.00/350k [00:00<?, ?B/s]

simplified/test-00000-of-00001.parquet:   0%|          | 0.00/347k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/43410 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5426 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5427 [00:00<?, ? examples/s]

{'anger': 5579,
 'disgust': 793,
 'fear': 726,
 'joy': 17410,
 'sadness': 3263,
 'surprise': 5367,
 'curiosity': 2191,
 'disappointment': 1269}

## Prefix Ablation

In [9]:
from sklearn.linear_model import RidgeClassifier, LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

PREFIXES = ['classification: ', 'clustering: ', 'search_document: ']
ABLATE_TARGETS = ['anger', 'curiosity']
rng = np.random.default_rng(ge.SEED)
sub = rng.choice(len(train_texts), size=8000, replace=False)
sub_texts = [train_texts[i] for i in sub]

ablation = {}
for pfx in PREFIXES:
    t2e = get_local_embeddings(sub_texts, model=ge.EMBEDDER, nomic_prefix=pfx,
                               cache_name='ablate_' + pfx.strip().strip(':'),
                               batch_size=128, show_progress=False)
    X = np.stack([t2e[t] for t in sub_texts]).astype(np.float32)
    aucs = {}
    for t in ABLATE_TARGETS:
        y = train_targets[t][sub]
        Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=ge.SEED, stratify=y)
        clf = RidgeClassifier().fit(Xtr, ytr)
        aucs[t] = float(roc_auc_score(yte, clf.decision_function(Xte)))
    ablation[pfx.strip()] = aucs
    del t2e, X; ge.clear_mem()

BEST_PREFIX = max(PREFIXES, key=lambda p: np.mean(list(ablation[p.strip()].values())))
ge.log_json('02_prefix_ablation', {'auc': ablation, 'best': BEST_PREFIX, 'used': ge.NOMIC_PREFIX})
if BEST_PREFIX != ge.NOMIC_PREFIX:
    print(f'ablation favors {BEST_PREFIX!r}; set NOMIC_PREFIX in ge_pipeline.py and rerun to switch')
ablation, BEST_PREFIX

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/445k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

Loaded model nomic-ai/modernbert-embed-base to cuda
Saved 7993 embeddings to /content/drive/MyDrive/hypothesaes_goemotions/emb_cache/ablate_clustering/chunk_000.npy


Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

Loaded model nomic-ai/modernbert-embed-base to cuda
Saved 7993 embeddings to /content/drive/MyDrive/hypothesaes_goemotions/emb_cache/ablate_search_document/chunk_000.npy


({'classification:': {'anger': 0.8617843186321494,
   'curiosity': 0.9282109890109891},
  'clustering:': {'anger': 0.828120007995077, 'curiosity': 0.8466461538461539},
  'search_document:': {'anger': 0.8388346703053711,
   'curiosity': 0.8568720879120878}},
 'classification: ')

## Embeddings

In [10]:
t2e = get_local_embeddings(train_texts + val_texts + test_texts, model=ge.EMBEDDER,
                           nomic_prefix=ge.NOMIC_PREFIX,
                           cache_name='goemotions_modernbert', batch_size=128)
train_emb = np.stack([t2e[t] for t in train_texts]).astype(np.float32)
val_emb   = np.stack([t2e[t] for t in val_texts]).astype(np.float32)
del t2e; ge.clear_mem()
train_emb.shape, val_emb.shape

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

Loaded model nomic-ai/modernbert-embed-base to cuda


Processing chunks:   0%|          | 0/2 [00:00<?, ?it/s]

Chunk 0:   0%|          | 0/391 [00:00<?, ?it/s]

Saved 49763 embeddings to /content/drive/MyDrive/hypothesaes_goemotions/emb_cache/goemotions_modernbert/chunk_000.npy


Chunk 1:   0%|          | 0/34 [00:00<?, ?it/s]

Saved 4258 embeddings to /content/drive/MyDrive/hypothesaes_goemotions/emb_cache/goemotions_modernbert/chunk_001.npy


((43410, 768), (5426, 768))

## Signal Gate

In [11]:
gate = {}
for t in ge.TARGETS:
    ytr, yva = train_targets[t], val_targets[t]
    if ytr.sum() < 20 or yva.sum() < 5:
        gate[t] = None
        continue
    clf = LogisticRegression(max_iter=2000).fit(train_emb, ytr)
    gate[t] = float(roc_auc_score(yva, clf.predict_proba(val_emb)[:, 1]))
ge.log_json('03_signal_gate', {'val_auc': gate})
gate

{'anger': 0.8802651855419145,
 'disgust': 0.9365096254108525,
 'fear': 0.969187675070028,
 'joy': 0.9005032507613121,
 'sadness': 0.9119019979226493,
 'surprise': 0.8687816374587513,
 'curiosity': 0.9304151247835133,
 'disappointment': 0.8587698121741199}

## Train SAE

In [12]:
M, K = 1024, 8
CKPT = os.path.join(ge.CKPT_DIR, 'goemotions_M1024_K8')
sae = train_sae(embeddings=train_emb, M=M, K=K, checkpoint_dir=CKPT,
                val_embeddings=val_emb, n_epochs=100)

  0%|          | 0/100 [00:00<?, ?it/s]

Early stopping triggered after 54 epochs
Saved model to /content/drive/MyDrive/hypothesaes_goemotions/checkpoints/goemotions_M1024_K8/SAE_M=1024_K=8.pt


## SAE Health

In [13]:
acts = sae.get_activations(val_emb)
dead_ratio = float((acts.sum(axis=0) == 0).mean())
mean_l0 = float((acts > 0).sum(axis=1).mean())
ge.log_json('04_sae', {'M': M, 'K': K, 'dead_neuron_ratio': dead_ratio,
                       'mean_active_per_example': mean_l0})
del acts; ge.clear_mem()
{'dead_neuron_ratio': dead_ratio, 'mean_active_per_example': mean_l0}

Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

{'dead_neuron_ratio': 0.0068359375, 'mean_active_per_example': 8.0}